# Chapter 5 (part 1) — Naive Bayes and LDA: Two Quick Churn Baselines

Covers: Naive Bayes → Discriminant Analysis (LDA), on the **churn** data, plus your first feature-engineering step.

**Chapter 5 project question: who will churn, and who should a limited retention budget target?** The data: IBM's Telco customer churn set, 7,043 customers with their contract, tenure, monthly and total charges, and which services they pay for; 26.5% churned. Every Chapter 5–6 notebook uses the same fixed train/test split from `load_churn_split()` (4,930 / 2,113 customers), so every model you build is compared fairly against the ones before it.

This notebook is a short warm-up (~40 min): you build a few features, then two fast baselines that later models must beat. Accuracy at a 0.5 cutoff is the only scorecard for now; notebook 15 replaces it with metrics built for this problem.

**How to work each section:** write your prediction **before** running anything, do the code, then write 2-3 bullets on what matched and what didn't. Feature engineering is yours; the encoding and scaling plumbing is pre-filled. The exam for this notebook is appended to `reviews/R01_regression-diagnostics.ipynb` (Q6–Q8).

In [1]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import CategoricalNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

pd.set_option("display.max_columns", None)

from mlstats.data import load_churn_split

train, test = load_churn_split()   # fixed stratified split; customerID already dropped
y_train, y_test = train["churned"].to_numpy(), test["churned"].to_numpy()

ADDONS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]


def report(p, y, label):
    # pre-filled scorecard: accuracy at a 0.5 cutoff vs. predicting "everyone stays"
    pred = p >= 0.5
    print(f"{label:28s} accuracy {np.mean(pred == y):.3f} (everyone-stays baseline {1 - y.mean():.3f}) | "
          f"churners caught {pred[y == 1].mean():.1%} | customers flagged {pred.sum()} of {len(y)}")


def confidence_table(p, y):
    # pre-filled: when the model says X%, how often do those customers actually churn?
    bands = pd.cut(p, [0, 0.1, 0.5, 0.9, 1.0], include_lowest=True)
    return (pd.DataFrame({"band": bands, "p": p, "churned": y})
              .groupby("band", observed=True)
              .agg(share_of_customers=("p", lambda s: len(s) / len(p)), mean_predicted=("p", "mean"), actual_churn_rate=("churned", "mean"))
              .round(3))


print(train.shape, test.shape, "| train churn rate:", round(y_train.mean(), 4))

(4930, 21) (2113, 21) | train churn rate: 0.2653


## 1. Build Your Features

Covers: *supplementary* (feature engineering; the book assumes it is done). A model only sees the columns you hand it, so turning raw columns into ones that carry the signal more directly is often the biggest lever you have. Three features, each from a relationship in the raw data:

- **`n_addons`**: how many of the six add-on services (`ADDONS`) the customer has. A fiber customer with TechSupport and OnlineSecurity has 2. Six correlated yes/no columns become one count.
- **`tenure_band`**: tenure binned into `0-12`, `13-24`, `25-48`, `49-72` months, so Naive Bayes (which needs categories) can use it.
- **`charge_drift`**: current monthly bill minus lifetime average monthly bill. Numeric picture: 10 months in with `TotalCharges` = $700 means a $70 average; if `MonthlyCharges` is now $80, drift = +$10 (a price increase or an added service). Drift = −$10 means a downgrade or discount.

Two rules that matter in every future notebook: write one function and apply it to train and test alike, and compute nothing from the test set (fixed bin edges here are fine; edges taken from quantiles would have to come from train only). Add a feature of your own if you like; append its name to the column lists in sections 2–3 to test it.

**Predict first:** (a) as `n_addons` goes from 0 to 6, does the churn rate fall steadily, or rise first and then fall? (b) Do customers whose bill drifted **up** by more than $2 churn more than those whose bill drifted **down** by more than $2 (by over 5 points), about the same, or less?

*Tools for this question:*
- `(df[ADDONS] == "Yes").sum(axis=1)` → count of "Yes" per row. Gotcha: "No internet service" is not "Yes".
- `pd.cut(x, bins=[0, 12, 24, 48, 72], labels=["0-12", "13-24", "25-48", "49-72"], include_lowest=True)` → categorical. Without `include_lowest=True`, tenure 0 becomes NaN.
- `TotalCharges / tenure` at tenure 0 gives NaN or inf (11 brand-new customers). Define their drift as 0, e.g. with `np.where(df["tenure"] > 0, ..., df["MonthlyCharges"])` for the average.
- Start from `out = df.copy()` and return `out`; never modify the input in place.

_Prediction (one line, before running anything):_
* My prediction is that as addon amount increases, the churn rate might fall steadily. I would assume that adding more addons might result in higher switching costs for the consumer so they are less likely to churn.
* I would assume that a bill drifting up by 2$ is too small of a increase to attribute it to a addon product (which I would then say less churn likelihood) and makes me think of it as a recent price increase. I would then assume that these consumers are more likely to churn then the ones that drifted down.

In [35]:
def add_features(df):
    """Return a copy of df with three new columns: n_addons, tenure_band, charge_drift."""
    out = df.copy()
    # Addon Count
    out["addon_count"] = (out[ADDONS] == "Yes").sum(axis = 1)

    # tenure band
    out["tenure_band"] = pd.cut(out["tenure"], bins = [0,12,24,48,72], labels = ["0-12", "13-24", "25-48", "49-72"], include_lowest=True)

    # drift
    out["charge_drift"] = np.where(out["tenure"] > 0, (out["MonthlyCharges"] - (out["TotalCharges"]/out["tenure"])), 0)

    return out

In [36]:
train_f = add_features(train)  # add_features applied to train
test_f = add_features(test)   # add_features applied to test

In [37]:
# pre-filled: sanity checks + churn rate by each new feature (consumes your train_f / test_f)
for d in (train_f, test_f):
    assert {"addon_count", "tenure_band", "charge_drift"} <= set(d.columns), "missing a feature"
    assert d["tenure_band"].notna().all(), "tenure_band has NaN (include_lowest?)"
    assert np.isfinite(d["charge_drift"]).all(), "charge_drift has NaN/inf (tenure == 0?)"

print(train_f.groupby("tenure_band", observed=True)["churned"].agg(churn_rate="mean", customers="size").round(3), "\n")
print(train_f.groupby("addon_count")["churned"].agg(churn_rate="mean", customers="size").round(3), "\n")
drift_bucket = pd.cut(train_f["charge_drift"], [-np.inf, -2, 2, np.inf], labels=["down > $2", "within $2", "up > $2"])
print(train_f.groupby(drift_bucket, observed=True)["churned"].agg(churn_rate="mean", customers="size").round(3))

             churn_rate  customers
tenure_band                       
0-12              0.477       1537
13-24             0.273        707
25-48             0.199       1118
49-72             0.101       1568 

             churn_rate  customers
addon_count                       
0                 0.204       1538
1                 0.467        658
2                 0.371        733
3                 0.274        773
4                 0.230        612
5                 0.130        399
6                 0.046        217 

              churn_rate  customers
charge_drift                       
down > $2          0.347        747
within $2          0.231       3409
up > $2            0.339        774


_What matched, what surprised you, and why (2-3 bullets):_
* **Surprise, not a match:** the addon churn rate does *not* fall steadily. It jumps from 20.4% at 0 add-ons to **46.7% at 1 add-on**, then falls steadily to 4.6% at 6. The 0-add-on group is 69% "No internet service" customers (cheap phone-only plans that rarely churn), so it is not "customers with nothing to be locked into". The falling part from 1 to 6 does fit your switching-cost story, but it is a correlation: heavy add-on customers also skew toward long tenure and longer contracts, so it can't separate lock-in from "loyal customers buy more".
* **Drift: wrong on the direction, but drift itself matters.** Customers whose bill drifted up by more than $2 churn at **33.9%**, about the same as those whose bill drifted down by more than $2 (**34.7%**). Both churn well above the "within $2" group (**23.1%**, which is 69% of customers). So it's not "price increases drive churn". Any change in the bill (a price increase, an added service, a downgrade or a discount ending) goes with more churn. A stable bill goes with customers who stay.

## 2. Naive Bayes

Covers: *Naive Bayes* (book). The exact Bayesian answer to "how likely is churn for a customer with these traits?" is the churn rate among past customers with **exactly** those traits. With 16 categorical columns, few customers match exactly, so Naive Bayes multiplies one-clue-at-a-time evidence instead. Numeric picture: if 80% of churners but 40% of stayers have clue A, seeing A doubles the churn odds. If clue B also has a 2-to-1 ratio, the model doubles the odds again, even when B mostly repeats A. That is the "naive" part: it assumes the clues are independent within each class, so correlated clues get counted more than once.

(a) **By hand, once:** for the profile *Contract = Month-to-month, InternetService = Fiber optic, PaymentMethod = Electronic check*, compute `p_naive` from `train_f` with the formula below, and `p_exact` = the churn rate among training customers who match all three.
(b) **Library:** fit `CategoricalNB` twice: on all the categorical columns plus your two categorical features (`cols_all`), and on a lean set where your `n_addons` replaces the six add-on columns (`cols_lean`).

**Predict first:** (a) will `p_naive` come out higher than `p_exact`, about equal (within 3 points), or lower? (b) Will the all-columns model beat the 73.5% everyone-stays baseline on accuracy, and will the lean model do better or worse than it?

*Tools for this question:*
- Naive posterior: P(churn | x) = P(churn)·∏ P(xⱼ | churn) / [P(churn)·∏ P(xⱼ | churn) + P(stay)·∏ P(xⱼ | stay)].
- `(train_f.loc[train_f["churned"] == 1, "Contract"] == "Month-to-month").mean()` → P(Contract = Month-to-month | churn).
- `CategoricalNB().fit(X, y)`; `.predict_proba(X)[:, 1]` → P(churn). Default `alpha=1.0` is Laplace smoothing (adds 1 to every count so a category never seen with churn doesn't zero out the whole product). `X` must be non-negative integer codes; the pre-filled `OrdinalEncoder` cell does that.

_Prediction (one line, before running anything):_

* I predict that naive may come out higher. I believe that the 3 features could be correlated meaning that they are counted more than twice when considering the probability. I believe the lean model will perform worse.

In [38]:
# naive conditions (to avoid a bunch of df.loc)
churn = train_f[train_f["churned"] == 1]
stay = train_f[train_f["churned"] == 0]

# clue churn and stay rates
p_mom_churn = (churn["Contract"] == "Month-to-month").mean()
p_fiber_churn = (churn["InternetService"] == "Fiber optic").mean()
p_echeck_churn = (churn["PaymentMethod"] == "Electronic check").mean()

p_mom_stay = (stay["Contract"] == "Month-to-month").mean()
p_fiber_stay = (stay["InternetService"] == "Fiber optic").mean()
p_echeck_stay = (stay["PaymentMethod"] == "Electronic check").mean()

# naive formula components
p_churn = train_f["churned"].mean()
p_stay = 1 - p_churn 

churn_side = p_churn * p_mom_churn * p_fiber_churn * p_echeck_churn
stay_side = p_stay * p_mom_stay * p_fiber_stay * p_echeck_stay


p_naive = churn_side/(churn_side + stay_side)

In [39]:
# non-naive bayesian classification
p_exact = train_f.loc[
    (train_f["Contract"] == "Month-to-month")
    & (train_f["InternetService"] == "Fiber optic")
    & (train_f["PaymentMethod"] == "Electronic check"),
    "churned",
].mean()


In [40]:
# pre-filled
print("naive Bayes:", round(p_naive, 3), "| exact:", round(p_exact, 3))

naive Bayes: 0.777 | exact: 0.606


In [41]:
# pre-filled: categorical columns -> integer codes (encoder fit on train only)
base_cats = ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines",
             "InternetService", *ADDONS, "Contract", "PaperlessBilling", "PaymentMethod"]
cols_all = base_cats + ["tenure_band", "addon_count"]
cols_lean = ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling", "SeniorCitizen", "tenure_band", "addon_count"]


def encode(cols):
    enc = OrdinalEncoder().fit(train_f[cols].astype(str))
    return enc.transform(train_f[cols].astype(str)), enc.transform(test_f[cols].astype(str))


Xtr_all, Xte_all = encode(cols_all)
Xtr_lean, Xte_lean = encode(cols_lean)
print(len(cols_all), "columns vs", len(cols_lean), "columns")

18 columns vs 7 columns


In [42]:
nb_all = CategoricalNB().fit(Xtr_all, y_train)     # CategoricalNB fit on Xtr_all, y_train
p_nb_all = nb_all.predict_proba(Xte_all)[:,1]   # its P(churn) on Xte_all
nb_lean = CategoricalNB().fit(Xtr_lean, y_train)     # CategoricalNB fit on Xtr_lean, y_train
p_nb_lean = nb_lean.predict_proba(Xte_lean)[:,1]    # its P(churn) on Xte_lean

In [43]:
# pre-filled: consumes p_nb_all, p_nb_lean
report(p_nb_all, y_test, "Naive Bayes, all columns")
report(p_nb_lean, y_test, "Naive Bayes, lean")
print("\nAll-columns model — does 'P = 97%' mean 97%?")
confidence_table(p_nb_all, y_test)

Naive Bayes, all columns     accuracy 0.713 (everyone-stays baseline 0.735) | churners caught 80.6% | customers flagged 950 of 2113
Naive Bayes, lean            accuracy 0.762 (everyone-stays baseline 0.735) | churners caught 66.0% | customers flagged 682 of 2113

All-columns model — does 'P = 97%' mean 97%?


,share_of_customers,mean_predicted,actual_churn_rate
band,,,
"(-0.001, 0.1]",0.450,0.009,0.076
"(0.1, 0.5]",0.101,0.270,0.174
"(0.5, 0.9]",0.133,0.734,0.273
"(0.9, 1.0]",0.316,0.974,0.561


_What matched, what surprised you, and why (2-3 bullets):_
* **Matched:** `p_naive` (0.777) is far above `p_exact` (0.606). The three clues (month-to-month, fiber, electronic check) are correlated with each other and all point toward churn, so Naive Bayes multiplies in the same evidence three times and overshoots by 17 points.
* **Lean prediction:** you predicted the lean model would do *worse*, but it was better on accuracy (0.762 vs 0.713 for all-columns, which is also below the 0.735 everyone-stays baseline). It caught fewer churners (66.0% vs 80.6%) because it flags fewer customers (682 vs 950). So your prediction was wrong on accuracy and right on churners caught.
* **Why:** the main thing the lean model removes is the six add-on columns, which say nearly the same thing and so get counted six times. That is the double-counting from the first bullet. Losing demographics like `Dependents` is probably a minor effect. The confidence table shows the damage: when the all-columns model says about 97%, only **56%** actually churn.

## 3. Linear Discriminant Analysis

Covers: *Discriminant Analysis* (book). LDA works on numeric features. Numeric picture with one feature: churners average about 18 months of tenure, stayers about 38, and tenure's spread within each group is about 24 months, so the two groups sit roughly 0.8 standard deviations apart. LDA finds the **weighted sum** of all the features that pushes the two group means furthest apart relative to the spread inside each group, assuming each group is a bell-shaped cloud with the same shape. With standardized inputs (pre-filled, scaler fit on train only), a weight says how far the score moves per 1-SD change in that feature, holding the others fixed, exactly like a coefficient in notebook 10.

Fit LDA on six numeric features, two of them yours: `tenure`, `MonthlyCharges`, `TotalCharges`, `SeniorCitizen`, `n_addons`, `charge_drift`.

**Predict first:** (a) will LDA beat the lean Naive Bayes on accuracy, and will it catch more or fewer churners? (b) Which feature gets the largest absolute weight: `tenure`, `MonthlyCharges`, or `TotalCharges`?

*Tools for this question:*
- `LinearDiscriminantAnalysis().fit(X, y)`; `.predict_proba(X)[:, 1]` → P(churn).
- `.coef_[0]` → one weight per feature, in the order of `num_cols` (comparable across features because the inputs are standardized).

_Prediction (one line, before running anything):_
* I believe that LDA could perform slightly better than lean. What concerns me is that it has less variables to help distinguish churners but I believe that its ability to find features that drive separation will also help it gain better performance.
* I believe tenure will be the biggest driver in separation.

In [44]:
# pre-filled: standardize numeric features (scaler fit on train only)
num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "addon_count", "charge_drift"]
scaler = StandardScaler().fit(train_f[num_cols])
Xtr_num, Xte_num = scaler.transform(train_f[num_cols]), scaler.transform(test_f[num_cols])

In [45]:
lda = LinearDiscriminantAnalysis().fit(Xtr_num, y_train)    # LinearDiscriminantAnalysis fit on Xtr_num, y_train
p_lda = lda.predict_proba(Xte_num)[:,1]  # its P(churn) on Xte_num

In [46]:
# pre-filled: consumes p_lda, lda, and section 2's p_nb_lean
report(p_nb_lean, y_test, "Naive Bayes, lean")
report(p_lda, y_test, "LDA, 6 numeric")
pd.Series(lda.coef_[0], index=num_cols, name="standardized weight").sort_values(key=abs, ascending=False).round(3)

Naive Bayes, lean            accuracy 0.762 (everyone-stays baseline 0.735) | churners caught 66.0% | customers flagged 682 of 2113
LDA, 6 numeric               accuracy 0.793 (everyone-stays baseline 0.735) | churners caught 51.9% | customers flagged 458 of 2113


MonthlyCharges    1.692
TotalCharges     -0.889
addon_count      -0.614
tenure           -0.423
SeniorCitizen     0.206
charge_drift     -0.041
Name: standardized weight, dtype: float64

_What matched, what surprised you, and why (2-3 bullets):_
* **Mostly matched:** LDA's accuracy (0.793) beats lean Naive Bayes (0.762), as you predicted. It catches fewer churners (51.9% vs 66.0%), but that mostly reflects where the 0.5 cutoff falls: LDA's probabilities are less inflated than Naive Bayes's, so fewer customers cross 0.5 (458 vs 682 flagged). The different feature sets are a smaller factor. Whether LDA *ranks* customers better is a question for notebook 15's ROC/lift.
* **Off on the largest weight:** it's `MonthlyCharges` (**+1.69**), not `tenure` (-0.42). `TotalCharges` comes next at -0.89, then `addon_count` -0.61, `SeniorCitizen` +0.21, and `charge_drift` is near zero at -0.04. Holding tenure and total spend fixed, a higher current bill is the strongest push toward churn. `charge_drift` adds almost nothing linear because the drift effect is U-shaped (both directions churn more), and a single straight-line weight can't capture a U.
* **Reading the weights:** `TotalCharges` ≈ tenure × `MonthlyCharges`, so the weights of those three features are partly offsetting each other. Read them as a group, not one at a time.

**Where this leaves the project question:** in one or two sentences, which of today's baselines would you hand the retention team, and what would you warn them about in Naive Bayes's probabilities?

In [47]:
train_f.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,churned,addon_count,tenure_band,charge_drift
5557,Female,0,No,No,5,Yes,No,Fiber optic,No,No,No,No,Yes,No,Month-to-month,No,Electronic check,80.20,384.25,No,0,1,0-12,3.350000
2270,Female,1,No,No,3,Yes,No,Fiber optic,No,No,Yes,No,Yes,No,Month-to-month,Yes,Electronic check,86.85,220.95,Yes,1,2,0-12,13.200000
6930,Female,0,Yes,No,3,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Credit card (automatic),75.15,216.75,Yes,1,0,0-12,2.900000
2257,Female,0,No,No,60,Yes,Yes,DSL,No,No,Yes,Yes,Yes,Yes,One year,No,Credit card (automatic),80.55,4847.05,No,0,4,49-72,-0.234167
898,Female,0,No,No,12,Yes,No,Fiber optic,Yes,No,No,Yes,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),98.90,1120.95,Yes,1,4,0-12,5.487500


* **Which baseline:** the all-columns Naive Bayes catches the most churners (80.6%) but has the lowest accuracy (0.713, under the everyone-stays baseline). Its probabilities can't be trusted as numbers: "97%" means 56%. If the retention team only needs a ranked list to call from the top, it may still work, but notebook 15's ROC/lift is needed to know if its ranking is any good. LDA (0.793 accuracy, probabilities less inflated) is the safer one to hand over today.
* **Warning:** Naive Bayes multiplies correlated clues as if they were independent, so its probabilities are pushed toward 0% and 100% (the confidence table). Ranking can survive that, but the retention budget decision (is this customer worth the offer cost?) needs real probabilities, and these aren't.